In [2]:
from training_utilities_2nd_part import *
from training_utilities import *
twitter_df = pd.read_csv("/Users/forough/PycharmProjects/mitigation3/Aiops_data_splitting_paper/code/Multivariate_time_series/twitter2/twitter3.csv")
twitter_df = twitter_df[4: len(twitter_df)-120]
twitter_df['Date'] = pd.to_datetime(twitter_df['Date'])
twitter_df['YearMonth'] = twitter_df['Date'].dt.to_period('M')
one_hot = pd.get_dummies(twitter_df['Symbol'], prefix='Symbol')
twitter_df = pd.concat([twitter_df, one_hot], axis=1)
twitter_df = twitter_df.drop(columns = ['Series', 'Trades', 'Symbol'])

,Date,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Deliverable Volume,%Deliverble,YearMonth,Symbol_ADANIPORTS,Symbol_MUNDRAPORT
0,2007-11-27,440.00,770.00,1050.00,770.00,959.0,962.90,984.72,27294366,2.687719e+15,9859619,0.3612,2007-11,False,True
1,2007-11-28,962.90,984.00,990.00,874.00,885.0,893.90,941.38,4581338,4.312765e+14,1453278,0.3172,2007-11,False,True
2,2007-11-29,893.90,909.00,914.75,841.00,887.0,884.20,888.09,5124121,4.550658e+14,1069678,0.2088,2007-11,False,True
3,2007-11-30,884.20,890.00,958.00,890.00,929.0,921.55,929.17,4609762,4.283257e+14,1260913,0.2735,2007-11,False,True
4,2007-12-03,921.55,939.75,995.00,922.00,980.0,969.30,965.65,2977470,2.875200e+14,816123,0.2741,2007-12,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3317,2021-04-26,725.35,733.00,739.65,728.90,729.2,730.75,733.25,9390549,6.885658e+14,838079,0.0892,2021-04,True,False
3318,2021-04-27,730.75,735.00,757.50,727.35,748.6,749.15,747.67,20573107,1.538191e+15,1779639,0.0865,2021-04,True,False
3319,2021-04-28,749.15,755.00,760.00,741.10,743.4,746.25,751.02,11156977,8.379106e+14,1342353,0.1203,2021-04,True,False
3320,2021-04-29,746.25,753.20,765.85,743.40,746.4,746.75,753.06,13851910,1.043139e+15,1304895,0.0942,2021-04,True,False


In [3]:
# twitter
# from variables_to_specify_twitter import *
columns_to_normalize = ['Prev Close', 'Open', 'High', 'Low', 'Last','Close', 'VWAP', 'Volume', 'Turnover', 'Deliverable Volume','%Deliverble']
twitter_target_col = 'Close'
forecast_avg_target_col_name = 'forecast_avg_Close'
avg_target_col_name = 'avg_Close'
No_of_datapoints_in_one_day = 1
date_col_name = 'Date'
one_month_window_size = 31
one_month_days =31
out_columns = ['Training dataset', 'Testing dataset', 'mae', 'mse', 'rmse', 'r2', 'mape', 'training_time',
              'Testing Error', 'testing_time']

twitter_drop_columnss = [twitter_target_col]+['YearMonth']+[date_col_name]
twitter_windows = [5,15,31,45,60,75,90]

print(twitter_target_col)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
twitter_df[columns_to_normalize] = scaler.fit_transform(twitter_df[columns_to_normalize])

twitter_df = twitter_df.dropna().reset_index(drop=True)

twitter_time_steps = 1
daily_df_avg = twitter_df[['Close']].copy()
daily_df_avg.rename(columns={'Close': 'avg_Close'}, inplace=True)

Close


,Date,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Deliverable Volume,%Deliverble,YearMonth,Symbol_ADANIPORTS,Symbol_MUNDRAPORT
0,2007-11-27,0.276794,0.550634,0.774216,0.570576,0.709167,0.712743,0.734103,0.279227,0.329318,0.439703,0.322305,2007-11,False,True
1,2007-11-28,0.712743,0.728634,0.724774,0.659896,0.647500,0.655217,0.697799,0.046763,0.052818,0.064606,0.274102,2007-11,False,True
2,2007-11-29,0.655217,0.666251,0.662766,0.631554,0.649167,0.647130,0.653161,0.052318,0.055733,0.047490,0.155346,2007-11,False,True
3,2007-11-30,0.647130,0.650447,0.698406,0.673638,0.684167,0.678269,0.687572,0.047054,0.052456,0.056023,0.226227,2007-11,False,True
4,2007-12-03,0.678269,0.691828,0.728895,0.701121,0.726667,0.718079,0.718129,0.030347,0.035202,0.036176,0.226884,2007-12,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3317,2021-04-26,0.514694,0.519859,0.518479,0.535277,0.517667,0.519196,0.523459,0.095984,0.084346,0.037155,0.024321,2021-04,True,False
3318,2021-04-27,0.519196,0.521522,0.533188,0.533946,0.533833,0.534537,0.535537,0.210436,0.188457,0.079169,0.021363,2021-04,True,False
3319,2021-04-28,0.534537,0.538158,0.535248,0.545755,0.529500,0.532119,0.538344,0.114063,0.102646,0.059657,0.058392,2021-04,True,False
3320,2021-04-29,0.532119,0.536660,0.540068,0.547730,0.532000,0.532536,0.540052,0.141645,0.127794,0.057985,0.029798,2021-04,True,False


# stationary

In [4]:
twitter_len_of_training_data_of_stationary_model =258
train = twitter_df[0:twitter_len_of_training_data_of_stationary_model] 
test = twitter_df[twitter_len_of_training_data_of_stationary_model:]
stationary_model = new_copied_lstm_statinary(twitter_df, twitter_len_of_training_data_of_stationary_model, twitter_target_col, twitter_drop_columnss, twitter_time_steps)

Epoch 1/10


2025-04-09 15:56:40.829970: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 199ms/step - loss: 0.1380
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0544
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0120
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0055
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0069
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0040
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0034
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0030
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0024
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0022
X_train shape: (258, 1, 10)
y_train shape: (258,)
X_train mean: 0.30701298
X_train std: 0.25963938
y_train mean: 0.44715407
y_train std: 0.20319735
y_train min: 0.12501563
y_train max: 1.0
total_test_error is : 0.0056053638
total_test_error_mae is:  0.06842276
total_time is:  5.04998675
train time is :  0
Model Type: Sequential
Storage Required: 0.08 MB
model storage

# Model reuse

Detected seasonality periods (ACF): [258 262 327 330 355 394 421 469 473 479 491]

In [5]:
# Model reuse
seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 258)
seasonality_periods_acf = seasonality_periods_acf_ls[0]

Detected seasonality periods (ACF): [258 262 327 330 355 394 421 469 473 479 491]
median_value is:  394


In [6]:
ratio_wass = len(filtered_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_tvd = len(filtered_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]
ratio_forecasted_wass = len(filtered_forecasted_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_forecasted_tvd = len(filtered_forecasted_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]

print('ratio_wass:', ratio_wass)
print('ratio_tvd:', ratio_tvd)
print('ratio_forecasted_wass:', ratio_forecasted_wass)
print('ratio_forecasted_tvd:', ratio_forecasted_tvd)

ratio_wass: 0.4166666666666667
ratio_tvd: 0.6666666666666666
ratio_forecasted_wass: 0.5
ratio_forecasted_tvd: 0.5833333333333334


## drift detection

In [7]:
df_copy = twitter_df[[twitter_target_col]]
target_col = twitter_target_col
time_steps = twitter_time_steps
df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = twitter_time_steps
x = 258* multiplier
window_len_=[x]
drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=twitter_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.


In [8]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage1 = new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model, 258, twitter_df, 'SA', twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_wass))

window is:  258
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 236ms/step - loss: 0.1554
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0646
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0154
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0053
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0070
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0041
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0037
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0033
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0027
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0023
i/window is :  1.0
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.2631
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1482
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0729
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0247
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0073
Epo

In [9]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage2 = new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model, 258, twitter_df, 'SA', twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_tvd))

window is:  258
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.1554
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0646
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0154
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0053
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0070
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0041
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0037
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0033
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0027
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0023
i/window is :  1.0
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.2631
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1482
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0729
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0247
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0073
Epoc

In [10]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage3 = new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model, 258, twitter_df, 'ES', twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_wass))

window is:  258
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1554
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0646
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0154
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0053
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0070
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0041
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0037
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0033
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0027
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0023
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.2631
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1482
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0729
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0247
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0073
Epoch 6/10
9/9 ━━━━━━━━

In [11]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage4 = new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model, 258, twitter_df, 'ES', twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_tvd))

window is:  258
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1554
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0646
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0154
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0053
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0070
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0041
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0037
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0033
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0027
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0023
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.2631
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1482
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0729
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0247
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0073
Epoch 6/10
9/9 ━━━━━━━━

In [13]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print("avg_ml_storage_reuse is : ", avg_ml_storage_reuse)

avg_ml_storage_reuse is :  0.08130264282226562


# informed retraining

In [14]:
lstm_informed_update(stationary_model,twitter_df, twitter_target_col, twitter_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices)

window is:  258
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1554
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0646
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0154
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0053
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0070
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0041
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0037
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0033
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0027
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0023
Model Type: Sequential
Storage Required: 0.08 MB
window is:  516
window is:  774
window is:  1032
window is:  1290
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 9.6263e-04
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 1.5990e-04
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 9.6811e-05
Epoch 4/10
9/9 ━━━━━━━━━

# periodical

In [8]:
mean_mse_per_window, min_index, mean_mae_per_window = periodical_lstm_training(twitter_df, twitter_target_col, twitter_time_steps, twitter_windows, twitter_drop_columnss)

window size is :  5
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.3438
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.3189
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.2949
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.2725
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.2524
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.2338
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.2163
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.1998
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1840
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1690
Model Type: Sequential
Storage Required: 0.08 MB
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 980ms/step - loss: 0.7783
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.7401
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.7032
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.6665
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━